# Transfer learning and fine tuning for image classification

## Transfer learning

### Importing the libraries

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import numpy as np
import cv2

from google.colab.patches import cv2_imshow
from google.colab import drive

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout

tf.__version__

### Loading the images

In [ ]:
drive.mount('/content/drive')

In [ ]:
path = '/content/drive/MyDrive/Cursos - recursos/Computer Vision Masterclass/Datasets/cat_dog_2.zip'
zip_object = zipfile.ZipFile(file=path, mode='r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
tf.keras.preprocessing.image.load_img('/content/cat_dog_2/training_set/cat/cat.1.jpg')

In [ ]:
tf.keras.preprocessing.image.load_img('/content/cat_dog_2/training_set/dog/dog.1.jpg')

### Train and test set

In [ ]:
training_generator = ImageDataGenerator(rescale=1./255,)
train_dataset = training_generator.flow_from_directory('/content/cat_dog_2/training_set',
                                                        target_size = (128, 128),
                                                        batch_size = 128,
                                                        class_mode = 'binary',
                                                        shuffle = True)

In [ ]:
test_generator = ImageDataGenerator(rescale=1./255)
test_dataset = test_generator.flow_from_directory('/content/cat_dog_2/test_set',
                                                     target_size = (128, 128),
                                                     batch_size = 1,
                                                     class_mode = 'binary',
                                                     shuffle = False)

### Pre-trained network

In [ ]:
base_model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=False,
                                               input_shape = (128,128,3))

In [ ]:
base_model.summary()

In [ ]:
len(base_model.layers)

In [ ]:
for layer in base_model.layers:
  layer.trainable = False

### Custom dense layer

In [ ]:
head_model = base_model.output
head_model = tf.keras.layers.GlobalAveragePooling2D()(head_model)
head_model = Dense(641, activation = 'relu')(head_model)
head_model = Dropout(0.2)(head_model)
head_model = Dense(641, activation = 'relu')(head_model)
head_model = Dropout(0.2)(head_model)
head_model = Dense(1, activation = 'sigmoid')(head_model)

### Building and training the neural network

In [ ]:
network = Model(inputs = base_model.input, outputs = head_model)

In [ ]:
(1280 + 2) / 2

In [ ]:
network.summary()

In [ ]:
network.compile(loss = 'binary_crossentropy', optimizer='Adam', 
                metrics = ['accuracy'])

In [ ]:
history = network.fit(train_dataset, epochs=5)

### Evaluating the neural network

In [ ]:
# Approach 1 (all pixels): 0.50
# Approach 2 (CNN): 0.72
# Approach 3 (Transfer learning): 0.96
network.evaluate(test_dataset)

## Fine tuning

### Implementing fine tuning

In [ ]:
base_model.trainable = True

In [ ]:
len(base_model.layers)

In [ ]:
fine_tuning_at = 100

In [ ]:
for layer in base_model.layers[:fine_tuning_at]:
  layer.trainable = False

In [ ]:
for layer in base_model.layers:
  print(layer, layer.trainable)

In [ ]:
network.compile(loss = 'binary_crossentropy', optimizer='Adam', metrics = ['accuracy'])

In [ ]:
history = network.fit(train_dataset, epochs=50)

In [ ]:
predictions = network.predict(test_dataset)
predictions

In [ ]:
predictions = (predictions > 0.5)
predictions

In [ ]:
test_dataset.classes

In [ ]:
# Approach 1 (all pixels): 0.50
# Approach 2 (CNN): 0.72
from sklearn.metrics import accuracy_score
accuracy_score(test_dataset.classes, predictions)

In [ ]:
network.evaluate(test_dataset)